# 🏠 House Price Prediction – Step by Step (Beginner Level)

**What we will do:**
1. Upload the CSV file
2. Look at the data
3. Clean the data
4. Make graphs
5. Train a simple model
6. Predict a house price

👉 Run each cell one by one (click the cell, press **Shift + Enter**).

## Step 1: Import the libraries
These are tools we need for data, graphs and prediction.

In [ ]:
import pandas as pd                 # for working with tables
import numpy as np                  # for numbers
import matplotlib.pyplot as plt     # for graphs
import seaborn as sns               # for nicer graphs

from sklearn.model_selection import train_test_split   # split data into train and test
from sklearn.linear_model import LinearRegression      # our prediction model
from sklearn.metrics import r2_score, mean_absolute_error

print("Libraries loaded ✅")

## Step 2: Upload the CSV file
Click **Choose Files** and select `house_price_db.csv`.

In [ ]:
from google.colab import files

uploaded = files.upload()      # a button will appear - choose house_price_db.csv

## Step 3: Read the file into a table (DataFrame)

In [ ]:
file_name = list(uploaded.keys())[0]   # get the name of the uploaded file
df = pd.read_csv(file_name)

print("File loaded:", file_name)
df.head()      # show first 5 rows

## Step 4: Understand the data

In [ ]:
print("Rows and columns:", df.shape)
print()
df.info()

In [ ]:
# Basic statistics (min, max, average...)
df.describe()

**Column meaning**

| Column | Meaning |
|---|---|
| date | Date of sale |
| price | Sale price (**this is what we want to predict**) |
| bedrooms | Number of bedrooms |
| grade | Quality grade of the house |
| has_basement | Basement? (True/False) |
| living_in_m2 | Living area in square metres |
| renovated | Was the house renovated? |
| nice_view | Does it have a nice view? |
| perfect_condition | Is it in perfect condition? |
| real_bathrooms | Number of bathrooms |
| has_lavatory | Has a lavatory? |
| single_floor | Is it a single-floor house? |
| month | Month of sale |
| quartile_zone | Price zone/area (1 to 4) |

# 🧹 DATA CLEANING

## Step 5: Check missing values
Missing values = empty cells. A model cannot learn from empty cells.

In [ ]:
print(df.isnull().sum())
print()
print("Total missing values:", df.isnull().sum().sum())

If everything shows **0**, there are no missing values 👍. (If not, you can fill them with `df['column'].fillna(df['column'].median(), inplace=True)`.)

## Step 6: Check and remove duplicate rows

In [ ]:
print("Duplicate rows before:", df.duplicated().sum())

df = df.drop_duplicates()          # remove duplicates
df = df.reset_index(drop=True)

print("Duplicate rows after :", df.duplicated().sum())
print("New shape:", df.shape)

## Step 7: Fix data types
- `date` should be a real date, not text.
- True/False columns should be 1/0 so the model can use them.

In [ ]:
# 1) Convert date text -> real date
df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year      # new column: year

# 2) Convert True/False -> 1/0
true_false_cols = ['has_basement', 'renovated', 'nice_view',
                   'perfect_condition', 'has_lavatory', 'single_floor']
for col in true_false_cols:
    df[col] = df[col].astype(int)

df.head()

## Step 8: Find outliers (very high / very low values)
A **boxplot** shows outliers as dots outside the box.

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
sns.boxplot(y=df['price'], color='skyblue')
plt.title('Price')

plt.subplot(1, 2, 2)
sns.boxplot(y=df['living_in_m2'], color='lightgreen')
plt.title('Living area (m²)')

plt.tight_layout()
plt.show()

Now remove extreme outliers using the **IQR rule** (a very common beginner method).

In [ ]:
def remove_outliers(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return data[(data[column] >= lower) & (data[column] <= upper)]

before = len(df)
df = remove_outliers(df, 'price')
df = remove_outliers(df, 'living_in_m2')
df = df.reset_index(drop=True)

print("Rows before:", before)
print("Rows after :", len(df))
print("Rows removed:", before - len(df))

## Step 9: Final check of the clean data

In [ ]:
print("Clean data shape:", df.shape)
print("Missing values:", df.isnull().sum().sum())
print("Duplicates:", df.duplicated().sum())
df.head()

# 📊 GRAPHS (Data Visualization)

## Graph 1: How are prices spread?

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df['price'], bins=40, kde=True, color='royalblue')
plt.title('Distribution of House Prices')
plt.xlabel('Price')
plt.ylabel('Number of houses')
plt.show()

## Graph 2: Living area vs Price
Bigger houses usually cost more.

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(x='living_in_m2', y='price', data=df, alpha=0.4)
plt.title('Living Area vs Price')
plt.xlabel('Living area (m²)')
plt.ylabel('Price')
plt.show()

## Graph 3: Average price by bedrooms, grade and zone

In [ ]:
plt.figure(figsize=(15, 4))

plt.subplot(1, 3, 1)
df.groupby('bedrooms')['price'].mean().plot(kind='bar', color='orange')
plt.title('Avg price by Bedrooms')
plt.ylabel('Average price')

plt.subplot(1, 3, 2)
df.groupby('grade')['price'].mean().plot(kind='bar', color='seagreen')
plt.title('Avg price by Grade')

plt.subplot(1, 3, 3)
df.groupby('quartile_zone')['price'].mean().plot(kind='bar', color='tomato')
plt.title('Avg price by Zone')

plt.tight_layout()
plt.show()

## Graph 4: Does a nice view / renovation increase price?

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
sns.barplot(x='nice_view', y='price', data=df, color='steelblue')
plt.title('Price: Nice view (0 = No, 1 = Yes)')

plt.subplot(1, 2, 2)
sns.barplot(x='renovated', y='price', data=df, color='seagreen')
plt.title('Price: Renovated (0 = No, 1 = Yes)')

plt.tight_layout()
plt.show()

## Graph 5: Correlation heatmap
Shows which columns are most related to **price**. Closer to **1** = strong positive relation.

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(df.drop(columns=['date']).corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

# Top factors affecting price
print(df.drop(columns=['date']).corr()['price'].sort_values(ascending=False))

# 🤖 PRICE PREDICTION

## Step 10: Choose inputs (X) and the answer (y)
- **X** = things we know about the house
- **y** = the price we want to predict

In [ ]:
X = df[['bedrooms', 'grade', 'has_basement', 'living_in_m2', 'renovated',
        'nice_view', 'perfect_condition', 'real_bathrooms', 'has_lavatory',
        'single_floor', 'quartile_zone']]
y = df['price']

print("X shape:", X.shape)
print("y shape:", y.shape)

## Step 11: Split into Training data (80%) and Testing data (20%)
The model learns from training data and is checked on testing data it has never seen.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print("Training rows:", len(X_train))
print("Testing rows :", len(X_test))

## Step 12: Train the model (Linear Regression)

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)      # the model learns here

print("Model trained ✅")

## Step 13: Check how good the model is

In [ ]:
y_pred = model.predict(X_test)

r2  = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"R² Score : {r2:.2f}   (closer to 1 = better)")
print(f"Average error (MAE): {mae:,.0f}   (on average the prediction is off by this much)")

## Step 14: Graph – Actual price vs Predicted price
If the dots follow the red line, the model is good.

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred, alpha=0.4)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')
plt.title('Actual vs Predicted Price')
plt.show()

## Step 15: Which features matter most?

In [ ]:
coef = pd.Series(model.coef_, index=X.columns).sort_values()

plt.figure(figsize=(8, 5))
coef.plot(kind='barh', color='purple')
plt.title('Effect of each feature on price')
plt.xlabel('Price change for +1 unit')
plt.show()

## Step 16: 🎯 Predict the price of YOUR house
Change the numbers below and run the cell.

In [ ]:
new_house = pd.DataFrame({
    'bedrooms': [3],
    'grade': [3],
    'has_basement': [1],          # 1 = Yes, 0 = No
    'living_in_m2': [200],
    'renovated': [0],
    'nice_view': [1],
    'perfect_condition': [0],
    'real_bathrooms': [2],
    'has_lavatory': [1],
    'single_floor': [0],
    'quartile_zone': [3]          # zone 1 to 4
})

predicted_price = model.predict(new_house)[0]
print(f"🏡 Predicted price: {predicted_price:,.0f}")

## ✅ Done!
You uploaded the data, cleaned it, drew graphs and predicted house prices.

**Next ideas to improve:** try `RandomForestRegressor` instead of `LinearRegression` for better accuracy.